In [ ]:
import sys
sys.path.append("../")
import os
os.environ["KERAS_BACKEND"] = "torch"

In [ ]:
from fame.experiments import exp_A_1, exp_A_2, exp_A_2_no_overwrite, exp_B_no_overwrite
from keras.models import load_model
import numpy as np

# check robustness because of numerical approximation error between solvers
from fame.abstract_domain.utils import check_is_robust 

In [ ]:
from configs.cifar10_configs import get_dataset, dataset_to_numpy, means_np, stddevs_np
train_dataset, val_dataset = get_dataset(augment=False, get_train=True, get_val=True)
test_dataset = get_dataset(augment=False, get_train=False, get_val=False)

In [ ]:
DATASET = "CIFAR10"
MODEL = "cnn"
from configs.cifar10_configs import get_dataset, dataset_to_numpy, means_np, stddevs_np

means_avg = np.mean(means_np)
std_avg = np.mean(stddevs_np)
eps = (2./255.)/std_avg #0.03
channel = 3
data_format = "channels_last"
n_class = 10

date="28-11-26"

In [ ]:
# means_avg= 0.47336665
# std_avg= 0.2507

In [ ]:
"""
download and process CIFAR10 data.
"""
x_train, y_train = dataset_to_numpy(train_dataset, means_np, stddevs_np)
x_valid, y_valid = dataset_to_numpy(val_dataset, means_np, stddevs_np)
x_test, y_test = dataset_to_numpy(test_dataset, means_np, stddevs_np)

In [ ]:
x_test_flattened = np.reshape(x_test, (-1, 3072))
print(f"x_test shape (Normalisé, NHWC): {x_test.shape}")
print(f"x_test dtype: {x_test.dtype}")

In [ ]:
x_test_flattened[0].shape

In [ ]:
k_model = load_model("./models/resnet_2b_ported.keras")

In [ ]:
k_model.eval()

In [ ]:
# indices_init = [i for i in range(0,100)]
# def is_robust(j):
#     return check_is_robust(model=k_model, 
#                     input_sample=x_test_flattened[j], 
#                     eps=eps, 
#                     channel=channel, 
#                     data_format=data_format, 
#                     n_class=n_class,
#                     means=means_avg,
#                     stddev=std_avg)
# indices = [i for i in indices_init if not is_robust(i)]
# indices

In [ ]:
robust_eps003 = [6,  13,  15,  16,  19,  21,  23,  29,  34,  41,  44,  45,  50,  54,
         60,  73,  75,  79,  82,  84,  90,  92,  98,  99, 102, 103, 104, 105,
        116, 122, 123, 131, 133, 141, 142, 151, 153, 154, 157, 166, 175, 196,
        202, 204, 208, 209, 215, 216, 220, 222, 225, 231, 232, 235, 240, 243,
        244, 252, 257, 265, 272, 276, 280, 283, 285, 286, 288, 289, 290, 296,
        297, 298, 311, 315, 321, 330, 331, 333, 334, 338, 341, 344, 345, 348,
        353, 361, 362, 369, 371, 372, 374, 379, 381, 382, 386, 389, 390, 392,
        400, 406, 414, 415, 417, 425, 429, 431, 440, 442, 443, 447, 452, 460,
        462, 469, 471, 472, 475, 484, 486, 487, 489, 490, 493, 495, 497, 499,
        507, 510, 511, 512, 514, 516, 517, 519, 521, 523, 524, 527, 529, 533,
        540, 541, 542, 544, 546, 547, 560, 571, 572, 576, 581, 588, 590, 591,
        592, 600, 601, 604, 605, 608, 609, 610, 612, 613, 619, 622, 626, 643,
        654, 656, 662, 664, 666, 681, 691, 693, 696, 700, 709, 721, 723, 724,
        726, 732, 736, 738, 741, 743, 745, 747, 750, 753, 759, 763, 764, 765,
        772, 774, 782, 786, 789, 791, 800, 801, 803, 812, 813, 815, 823, 824,
        827, 828, 830, 832, 838, 839, 840, 842, 844, 847, 854, 856, 857, 858,
        859, 864, 868, 871, 872, 874, 879, 883, 885, 891, 894, 899, 902, 903,
        911, 914, 915, 917, 921, 925, 931, 934, 935, 939, 940, 946, 951, 955,
        958, 959, 960, 968, 973, 975, 984, 985, 989, 990, 994, 997, 999]

In [ ]:
#robust_eps003= [  6,  13,  15,  16,  19,  21,  23,  29,  34,  41,  44,  45,  50,  54, 60,  73,  75,  79,  82,  84,  90,  92,  98,  99, ]
indices = [i for i in range(0,1000) if i not in robust_eps003]
len(indices)

## Experiment A: MILP versus Greedy
### A.1 in one round: what is the running time and largest free set obtained when using milp or greedy

In [ ]:
dataframe_repository = "./results/CIFAR10/eps-{}/{}".format(f'{eps:.2f}'.replace('0.', ''),date)
os.makedirs(dataframe_repository, exist_ok =True)

In [ ]:
EXP = "A_1"
filename = "{}_{}_{}".format(DATASET, MODEL, EXP)
dataframe_filename_A1 = filename
dataframe_repository, filename

In [ ]:
dico_a_1 = exp_A_1(
        model=k_model,
        x_test=x_test_flattened,
        y_test=y_test,
        indices=indices, #new_indices
        eps=eps,
        dataframe_repository=dataframe_repository,
        dataframe_filename=dataframe_filename_A1,
        channel=channel,
        data_format=data_format,
        n_class=n_class,
        verbose=1,
        means=means_avg,
        stddev=std_avg
        #decomon_method=decomon_method
    )

### A.2 iteratively, what is the running time and largest free set obtained when using milp or greedy

In [ ]:
EXP = "A_2"
filename = "{}_{}_{}".format(DATASET, MODEL, EXP)
dataframe_filename_A2 = filename

In [ ]:
failing_indexes_a2 = []
for j in indices:
    try:
        dico_a_2 = exp_A_2_no_overwrite(
                model=k_model,
                x_test=x_test_flattened,
                y_test=y_test,
                indices=[j],#indices,
                eps=eps,
                dataframe_repository=dataframe_repository,
                dataframe_filename=dataframe_filename_A2,
                channel=channel,
                data_format=data_format,
                n_class=n_class,
                verbose=1,
                sleep_time=0,
                means=means_avg,
                stddev=std_avg
            )
    except Exception as ex:
        print("exception: ", ex)
        failing_indexes_a2.append(j)
        print("fail: ", failing_indexes_a2)
        continue

# B Distance 2 minimality

In [ ]:
EXP = "B"
filename = "{}_{}_{}".format(DATASET, MODEL, EXP)
dataframe_filename_B = filename

In [ ]:
failing_indexes_b = []
for j in indices:
    try:
        print("ongoing index", j)
        exp_B_no_overwrite(
            model=k_model,
            x_test=x_test_flattened,
            y_test=y_test,
            indices=[j],
            eps=eps,
            dataframe_repository=dataframe_repository,
            dataframe_filename=dataframe_filename_B,
            channel=channel,
            data_format=data_format,
            method="greedy",
            attack="fgsm",
            traversal_order="greedy",
            device='cuda',#"mps",
            n_class=n_class,
        )
    except Exception as ex:
        print("exception: ", ex)
        failing_indexes_b.append(j)
        print("fail: ", failing_indexes_b)
        continue

In [ ]:
# failing_indexes_b = [0, 1, 2, 3, 4, 5, 7, 9, 12, 17, 18, 20, 22, 24, 25, 26, 27, 30, 32, 33, 35, 36, 39, 40, 42, 43, 46, 47, 49, 51, 52, 53, 55, 56, 57, 58, 59, 62, 63, 64, 65, 66, 67, 68, 69, 71, 72, 74, 77, 78, 80, 81, 85, 86, 87, 88, 91, 93, 94, 95, 96]

In [ ]:
new_indices = [102, 103, 104, 105,
        116, 122, 123, 131, 133, 141, 142, 151, 153, 154, 157, 166, 175, 196,
        202, 204, 208, 209, 215, 216, 220, 222, 225, 231, 232, 235, 240, 243,
        244, 252, 257, 265, 272, 276, 280, 283, 285, 286, 288, 289, 290, 296,
        297, 298, 311, 315, 321, 330, 331, 333, 334, 338, 341, 344, 345, 348,
        353, 361, 362, 369, 371, 372, 374, 379, 381, 382, 386, 389, 390, 392,
        400, 406, 414, 415, 417, 425, 429, 431, 440, 442, 443, 447, 452, 460,
        462, 469, 471, 472, 475, 484, 486, 487, 489, 490, 493, 495, 497, 499,
        507, 510, 511, 512, 514, 516, 517, 519, 521, 523, 524, 527, 529, 533,
        540, 541, 542, 544, 546, 547, 560, 571, 572, 576, 581, 588, 590, 591,
        592, 600, 601, 604, 605, 608, 609, 610, 612, 613, 619, 622, 626, 643,
        654, 656, 662, 664, 666, 681, 691, 693, 696, 700, 709, 721, 723, 724,
        726, 732, 736, 738, 741, 743, 745, 747, 750, 753, 759, 763, 764, 765,
        772, 774, 782, 786, 789, 791, 800, 801, 803, 812, 813, 815, 823, 824,
        827, 828, 830, 832, 838, 839, 840, 842, 844, 847, 854, 856, 857, 858,
        859, 864, 868, 871, 872, 874, 879, 883, 885, 891, 894, 899, 902, 903,
        911, 914, 915, 917, 921, 925, 931, 934, 935, 939, 940, 946, 951, 955,
        958, 959, 960, 968, 973, 975, 984, 985, 989, 990, 994, 997, 999]

failing_newindexes_b = []
for j in new_indices:
    try:
        print("ongoing index", j)
        exp_B_no_overwrite(
            model=k_model,
            x_test=x_test_flattened,
            y_test=y_test,
            indices=[j],
            eps=eps,
            dataframe_repository=dataframe_repository,
            dataframe_filename=dataframe_filename_B,
            channel=channel,
            data_format=data_format,
            method="greedy",
            attack="fgsm",
            traversal_order="greedy",
            device='cuda',#"mps",
            n_class=n_class,
        )
    except Exception as ex:
        print("exception: ", ex)
        failing_newindexes_b.append(j)
        print("fail: ", failing_newindexes_b)
        continue

In [ ]:
# fail:  [102, 103, 104, 105, 116, 123, 133, 141, 142, 153, 154, 166, 175, 196, 202, 204, 208, 215, 216, 220, 231, 232, 235, 240, 244, 257, 265, 272, 276, 280, 283, 286, 288, 289, 290, 296, 297, 298, 311, 315, 321, 330, 331, 333, 334, 338, 341, 344, 345, 348, 353, 361, 362, 371, 374, 379, 382, 386, 389, 390, 392, 406, 414, 415, 417, 429, 440, 442, 447, 452, 460, 462, 469, 471, 472, 475, 484, 489, 490, 493, 495, 497, 499, 510, 511, 512, 514, 516, 517, 519, 521, 523, 524, 527, 529, 533, 540, 541, 542, 544, 547, 560, 571, 572, 576, 581, 588, 590, 591, 592, 600, 601, 604, 605, 608, 610, 612, 622, 626, 643, 654, 662, 664, 681, 691, 693, 696, 700, 709, 721, 723, 724, 726, 732, 738, 741, 743, 745, 747, 753, 759, 763, 764, 765, 772, 774, 782, 786, 789, 791, 800, 803, 812, 813, 815, 823, 824, 827, 828, 830, 832, 839, 840, 842, 844, 847, 854, 856, 858, 859, 864, 868, 871, 872, 874, 879, 883, 885, 891, 894, 899, 902, 903, 911, 914, 917, 921, 925, 931, 934, 935, 939, 946, 955, 958, 959, 968, 973, 975, 984, 985, 994, 997, 999]